In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_T': None,
  'anneal_steps': None,
  'anneal_interval': 100,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'dispcorr': True,
  'nonbondedmethod': 'PME'},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}},
 'validation': {}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", "NonbondedForce/epsilon", "NonbondedForce/sigma"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [6]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[10].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[11].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9]], target_charges=[0.0])

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble


Anisotropic pressure control


== Energy minimization ==


== Start Relaxation ==


#"Step","Potential Energy (kJ/mole)","Temperature (K)","Density (g/mL)","Speed (ns/day)","Time Remaining"


...

 78%|███████▊  | 78/100 [00:01<00:00, 47.50it/s]

 84%|████████▍ | 84/100 [00:02<00:00, 38.22it/s]

 91%|█████████ | 91/100 [00:02<00:00, 43.81it/s]

 97%|█████████▋| 97/100 [00:02<00:00, 36.16it/s]

100%|██████████| 100/100 [00:02<00:00, 40.46it/s]

In [9]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)

Initial neff: [30]


In [10]:
trainer.neff = [50]

In [11]:
trainer.fit(10, 2)

Pair list:   0%|          | 0/100 [00:00<?, ?it/s]

Pair list:   1%|          | 1/100 [00:00<00:22,  4.43it/s]

Pair list:   5%|▌         | 5/100 [00:00<00:07, 12.33it/s]

Pair list:  29%|██▉       | 29/100 [00:00<00:01, 70.58it/s]

Pair list:  39%|███▉      | 39/100 [00:00<00:00, 75.06it/s]

...

Get weight, Memory Usage: 9613.53 MB
Finish loss calc Memory Usage: 9626.91 MB


grad obtained.... Memory Usage: 315.91 MB
grad modify.... Memory Usage: 324.81 MB
update finished.... Memory Usage: 343.22 MB
Effective sample sizes:
  sample_0: 98
  Total: 99.98235410365288


Epoch 8, Loss: 0.9825316755796804, Memory Usage: 415.55 MB
Loss:  0.9825316755796804
Best Loss:  0.9825316755796804 at epoch  8
Epoch 8 completed in 56.80 seconds.
----
Init Energy calc: Memory Usage: 416.86 MB


Energy calc: 30.02 sec. Memory Usage: 6612.25 MB
Get weight, Memory Usage: 6655.42 MB
Finish loss calc Memory Usage: 6666.50 MB


grad obtained.... Memory Usage: 315.34 MB
grad modify.... Memory Usage: 326.14 MB
update finished.... Memory Usage: 347.94 MB
Effective sample sizes:
  sample_0: 87
  Total: 88.04430631948333
Loss:  0.970005172542695
Best Loss:  0.970005172542695 at epoch  9
Epoch 9 completed in 51.46 seconds.
----


In [12]:
trainer.get_loss_gradients()

Init Energy calc: Memory Usage: 373.30 MB


Energy calc: 27.69 sec. Memory Usage: 11459.59 MB
Get weight, Memory Usage: 11496.67 MB
Finish loss calc Memory Usage: 11508.33 MB


(Array(0.95749159, dtype=float64),
 {'HarmonicAngleForce': {'angle': Array([ 2.24244004, -0.80292254, -2.52023684, -0.21650571,  3.26204889,
           0.73617405, -3.79622637,  3.77136592,  3.04152445, -4.48097981],      dtype=float64),
   'k': Array([ 3.75538748e-04,  2.22882245e-04,  6.33580765e-04, -6.93007111e-05,
           4.73095914e-04,  9.68498813e-05,  4.86899265e-04, -2.86058989e-04,
          -4.44987259e-05,  5.75597296e-04], dtype=float64)},
  'HarmonicBondForce': {'k': Array([-1.95238008e-07,  3.95467754e-07, -2.16618610e-07, -5.16839704e-10,
           3.19068626e-07,  5.06202471e-08,  1.95857427e-07], dtype=float64),
   'length': Array([-109.23171483,   42.08131842,   59.5878975 ,   -0.92995664,
            -9.6502523 ,  -34.7231794 ,   24.22472106], dtype=float64)},
  'NonbondedForce': {'charge': Array([ 0.54711782,  1.38479101,  0.78372408,  1.6297911 , -1.25391966,
          -2.8810916 ,  1.89797124,  1.89765281, -3.12027069,  1.8404173 ,
          -0.84958254,  1.

In [13]:
trainer.neff = neff_prev

In [14]:
# Shortened for the example (the production run used fit(490, 2)).
trainer.fit(30, 2)

Init Energy calc: Memory Usage: 280.45 MB


Energy calc: 28.94 sec. Memory Usage: 11376.59 MB
Get weight, Memory Usage: 11414.42 MB
Finish loss calc Memory Usage: 11426.20 MB


grad obtained.... Memory Usage: 304.84 MB
grad modify.... Memory Usage: 315.80 MB
update finished.... Memory Usage: 337.92 MB
Effective sample sizes:
  sample_0: 55
  Total: 56.82971904925585


Epoch 10, Loss: 0.9574915893797029, Memory Usage: 407.55 MB
Loss:  0.9574915893797029
Best Loss:  0.9574915893797029 at epoch  10
Epoch 10 completed in 50.63 seconds.
----
Init Energy calc: Memory Usage: 409.83 MB


Energy calc: 28.86 sec. Memory Usage: 12288.86 MB
Get weight, Memory Usage: 12334.20 MB
Finish loss calc Memory Usage: 12345.27 MB


...

 86%|████████▌ | 86/100 [00:02<00:00, 43.14it/s]

 93%|█████████▎| 93/100 [00:02<00:00, 47.47it/s]

 99%|█████████▉| 99/100 [00:02<00:00, 30.98it/s]

100%|██████████| 100/100 [00:02<00:00, 37.33it/s]

Loss:  0.73585842890229
Best Loss:  0.73585842890229 at epoch  39
Epoch 39 completed in 422.85 seconds.
----


In [15]:
# Shortened for the example (the production run used fit(40, 2)).
trainer.fit(10, 2)

Pair list:   0%|          | 0/100 [00:00<?, ?it/s]

Pair list:   1%|          | 1/100 [00:00<00:32,  3.00it/s]

Pair list:   7%|▋         | 7/100 [00:00<00:04, 19.47it/s]

Pair list:  31%|███       | 31/100 [00:00<00:00, 77.73it/s]

Pair list:  42%|████▏     | 42/100 [00:00<00:00, 80.90it/s]

...

 83%|████████▎ | 83/100 [00:02<00:00, 49.30it/s]

 89%|████████▉ | 89/100 [00:02<00:00, 33.03it/s]

 96%|█████████▌| 96/100 [00:02<00:00, 39.15it/s]

100%|██████████| 100/100 [00:02<00:00, 39.37it/s]

Loss:  0.6447926393140366
Best Loss:  0.6447926393140366 at epoch  49
Epoch 49 completed in 417.21 seconds.
----


In [16]:
# To resume later from the checkpoint (it records the force fields, the
# sampling and the targets; the hooks have to be registered again):
# trainer = ThermodynamicTrainer.from_checkpoint("train_state_2rdf_1adf_density.pkl", setup=False)
# trainer.add_modifyfn("after_grad", grad_modify)
# trainer.add_modifyfn("after_update", ffparams_modify)
# trainer.setup()
# trainer.fit(100, 2)